## Fix the missing pLDDT score of the domainome .pdb files

Purpose: add_caps.py used mdtraj package, which accidentally erased the pLDDT score in the b-factor column of the AF2 models. We need to add them back to be able to use pLDDT for downstream analysis. 

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.PDBIO import PDBIO


repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]

os.chdir(repo_root)

# Path to domainome pdb directory
domainome_pdb_dir = "/scratch/ymeng/DPAM/dpam_domaindome_masif_db/data_preparation/01-benchmark_pdbs"

# Path to domainome info .csv
domainome_info_csv = "/scratch/ymeng/DPAM/dpam_domaindome_masif_db/DPAM_AI_AFDB_domainome_v6_domains_info_metrics.csv"

# Path to write intermediate files
intermediate_dir = "/scratch/ymeng/DPAM/data/fix_pLDDT"
raw_pdb_dir = os.path.join(intermediate_dir, "raw_pdb")           # Directory for afdb downloaded pdbs
domain_plddt_dir = os.path.join(intermediate_dir, "domain_plddt_pdb") # Directory to save domainome pdb files with pLDDT scores
domain_info_dir = os.path.join(intermediate_dir, "domain_info")   # Directory to save domainome info .csv with pLDDT scores

os.makedirs(raw_pdb_dir, exist_ok=True)
os.makedirs(domain_plddt_dir, exist_ok=True)
os.makedirs(domain_info_dir, exist_ok=True)

# Load pdb parser
parser = PDBParser()

# Load pdb io
io = PDBIO()

In [5]:
# Read domainome info .csv
df_domainome_info = pd.read_csv(domainome_info_csv)

df_domainome_info.iloc[0]

id                                                          Q8WVQ1-F1-nD1_A
uniprot_accn                                                         Q8WVQ1
Domain                                                                  nD1
Range                                                                86-400
ECOD_num                                                               2167
ECOD_key                                                            e1s1dA1
T-group                                                               5.1.2
DPAM_prob                                                             0.979
HH_prob                                                                 1.0
DALI_zscore                                                            57.6
Hit_cov                                                               0.994
Tgroup_cov                                                            0.969
Judge                                                           good_domain
Hcount      

In [6]:
# Get all unique uniprot_accn
all_uniprot_accn = df_domainome_info['uniprot_accn'].unique()

len(all_uniprot_accn)

18629

In [7]:
# ----- Helper functions -----
import requests

# Function to download the top AF2 database entry's .pdb and .json files for a given UniProt ID
def download_af2_model(uniprot_id, out_dir):
    """
    Downloads an AlphaFold2 (AF2) model by querying the AlphaFold DB API using the UniProt ID,
    and saves the corresponding PDB file and PAE .json file directly to out_dir using the filenames in the urls.

    Args:
        uniprot_id (str): The UniProt ID (e.g., 'Q8NB16').
        out_dir (str): The directory to save the downloaded PDB and JSON file.

    Returns:
        str or None: The URL to the downloaded PDB file or None if not found or download failed.
    """

    # Query the AlphaFold API to get pdbUrl and paeDocUrl
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    try:
        api_response = requests.get(api_url)
        if api_response.status_code != 200:
            print(f"Failed to query AlphaFold API for {uniprot_id} (HTTP {api_response.status_code})")
            return None
        api_json = api_response.json()
        if not api_json or 'pdbUrl' not in api_json[0] or 'paeDocUrl' not in api_json[0]:
            print(f"No pdbUrl or paeDocUrl found in AlphaFold API response for {uniprot_id}")
            return None
        pdb_url = api_json[0]['pdbUrl']
        paeDocUrl = api_json[0]['paeDocUrl']
    except Exception as e:
        print(f"Error fetching/parsing AlphaFold API response: {e}")
        return None

    try:
        os.makedirs(out_dir, exist_ok=True)

        # Download the PDB file
        pdb_response = requests.get(pdb_url)
        if pdb_response.status_code != 200:
            print(f"Failed to download PDB file for {uniprot_id} at {pdb_url} (HTTP {pdb_response.status_code})")
            return None
        pdb_filename = f"{uniprot_id}.pdb"
        save_path_pdb = os.path.join(out_dir, pdb_filename)
        with open(save_path_pdb, "wb") as f:
            f.write(pdb_response.content)

        return pdb_filename
    except Exception as e:
        print(f"Failed to download or save PDB or PAE file for {uniprot_id}: {e}")
        return None

# pdb_filename = download_af2_model("Q8NB16", "/scratch/ymeng/DPAM/results/test")

In [8]:
# ----- Function to map pLDDT scores -----
import numpy as np
import copy

def map_plddt_scores(domain_struct, af2_fl_struct, max_match_distance=1e-2, verbose=False):
    """
    Copy pLDDT scores from af2_fl_struct to a deepcopy of domain_struct by CA nearest-neighbor matching.
    
    Args:
        domain_struct: Bio.PDB structure object for the domain (to copy to).
        af2_fl_struct: Bio.PDB structure object with pLDDT stored in CA B-factor (to copy from).
        max_match_distance: Maximum allowed CA-CA distance (Angstroms) for considering a match.
        verbose: If True, print summary statistics and one example of failed matches.
        
    Returns:
        domain_struct_with_plddt: deep copy of domain_struct with updated pLDDT in B-factor field
        match_stats: dict with n_matched, n_failed, max_matched_dist, failed_matches
    """
    # Copy domain_struct so we don't overwrite the original
    domain_struct_with_plddt = copy.deepcopy(domain_struct)

    # Collect all CA atoms from af2_fl_struct into a list
    af2_ca_atoms = []
    for model in af2_fl_struct:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.get_id() == 'CA':
                        af2_ca_atoms.append(atom)
            break
        break

    # Also collect all CA atoms from domain_struct_with_plddt into a list
    domain_ca_atoms = []
    for model in domain_struct_with_plddt:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.get_id() == 'CA':
                        domain_ca_atoms.append(atom)
            break
        break

    if verbose:
        print(f"Number of CA atoms in domain: {len(domain_ca_atoms)}")
        print(f"Number of CA atoms in af2 full: {len(af2_ca_atoms)}")

    # Build arrays for nearest-neighbor coordinate matching
    af2_ca_coords = np.array([atom.get_coord() for atom in af2_ca_atoms], dtype=float)
    af2_ca_bfactors = np.array([atom.get_bfactor() for atom in af2_ca_atoms], dtype=float)

    failed_matches = []
    match_distances = []

    for domain_ca in domain_ca_atoms:
        domain_coord = np.array(domain_ca.get_coord(), dtype=float)
        residue = domain_ca.get_parent()

        # Brute-force nearest neighbor in AF2 CA list
        deltas = af2_ca_coords - domain_coord
        dists = np.linalg.norm(deltas, axis=1)
        nearest_idx = int(np.argmin(dists))
        nearest_dist = float(dists[nearest_idx])

        if nearest_dist <= max_match_distance:
            plddt_value = float(af2_ca_bfactors[nearest_idx])
            for atom in residue:
                atom.set_bfactor(plddt_value)
            match_distances.append(nearest_dist)
        else:
            # If nearest CA is still too far, mark as unmatched
            for atom in residue:
                atom.set_bfactor(0.0)
            failed_matches.append((tuple(domain_coord), nearest_dist))
    
    stats = {
        "n_matched": len(match_distances),
        "n_failed": len(failed_matches),
        "max_matched_dist": max(match_distances) if match_distances else None,
        "failed_matches": failed_matches,
    }

    if verbose:
        print(
            f"Copied pLDDT scores to domain_struct_with_plddt. "
            f"Matched={stats['n_matched']}, Failed={stats['n_failed']}, "
            f"max_matched_dist={stats['max_matched_dist']}"
        )
        if failed_matches:
            print("Example failed match (coord, nearest_dist):", failed_matches[0])

    return domain_struct_with_plddt, stats

# Example usage:
# domain_struct_with_plddt, match_stats = map_plddt_scores(domain_struct, af2_fl_struct)

In [9]:
# ----- Full workflow for each uniprot_accn -----

def map_plddt_for_uniprot(uniprot_accn, df_domainome_info):
    """
    Process a single uniprot_accn: copies pLDDT scores onto each domain PDB,
    writes new PDBs, and updates a DataFrame with match stats.

    Args:
        uniprot_accn (str): The UniProt accession to process.
        df_domainome_info (pd.DataFrame): DataFrame containing all domain info.

    Returns:
        pd.DataFrame: Updated DataFrame for this uniprot_accn with pLDDT mapping statistics.
    """
    # Subset df_domainome_info for the current uniprot_accn
    df_domainome_info_uniprot = df_domainome_info[df_domainome_info['uniprot_accn'] == uniprot_accn]

    # Download the AF2 model
    pdb_filename = download_af2_model(uniprot_accn, raw_pdb_dir)
    af2_pdb_path = os.path.join(raw_pdb_dir, pdb_filename)

    # Load the AF2 model
    af2_fl_struct = parser.get_structure("AF2", af2_pdb_path)

    # Initialize a df to store the updated rows
    df_domainome_info_uniprot_plddt = pd.DataFrame()

    for idx, row in df_domainome_info_uniprot.iterrows():

        # ---- Workflow for individual domainome_id ----
        domainome_id = row['id']

        # Path to the domainome pdb file
        domainome_pdb_path = os.path.join(domainome_pdb_dir, f"{domainome_id}.pdb")

        # Load the domainome pdb file
        domain_struct = parser.get_structure("domainome", domainome_pdb_path)

        # Map pLDDT scores from AF2 to domain_struct
        domain_struct_with_plddt, match_stats = map_plddt_scores(domain_struct, af2_fl_struct)

        # Save the domain_struct_with_plddt to a new pdb file using Bio.PDB's PDBIO
        domain_struct_with_plddt_path = os.path.join(domain_plddt_dir, f"{domainome_id}.pdb")
        io.set_structure(domain_struct_with_plddt)
        io.save(domain_struct_with_plddt_path)

        # Add match_stats to 4 new columns in row
        row['n_matched'] = match_stats.get('n_matched')
        row['n_failed'] = match_stats.get('n_failed')
        row['max_matched_dist'] = match_stats.get('max_matched_dist')
        row['failed_matches'] = match_stats.get('failed_matches')

        # Append the updated row to df_domainome_info_uniprot_plddt
        df_domainome_info_uniprot_plddt = pd.concat([df_domainome_info_uniprot_plddt, pd.DataFrame([row])], ignore_index=True)

    # Save the updated df to a csv file
    df_domainome_info_uniprot_plddt.to_csv(os.path.join(domain_info_dir, f"{uniprot_accn}.csv"), index=False)

    return df_domainome_info_uniprot_plddt

# df_domainome_info_uniprot_plddt = map_plddt_for_uniprot(uniprot_accn, df_domainome_info)

In [10]:
# ----Workflow for each uniprot_accn----
uniprot_accn = all_uniprot_accn[0]

df_domainome_info_uniprot_plddt = map_plddt_for_uniprot(uniprot_accn, df_domainome_info)
df_domainome_info_uniprot_plddt

,id,uniprot_accn,Domain,Range,ECOD_num,ECOD_key,T-group,DPAM_prob,HH_prob,DALI_zscore,...,strand_frac,sasa,norm_sasa,protein_name,gene_name,domain_length,n_matched,n_failed,max_matched_dist,failed_matches
0,Q8WVQ1-F1-nD1_A,Q8WVQ1,nD1,86-400,2167,e1s1dA1,5.1.2,0.979,1.0,57.6,...,0.526984,13852.167474,43.697689,Soluble calcium-activated nucleotidase 1,CANT1,315,315,0,0.0,[]


In [11]:
# Split all_uniprot_accn based on the number of CPU cores, then parallelize the map_plddt_for_uniprot function
from joblib import Parallel, delayed
import numpy as np
from tqdm import tqdm
import multiprocessing

# Get the number of available CPU cores
n_cores = multiprocessing.cpu_count()
print(f"Splitting {len(all_uniprot_accn)} uniprot_accn into {n_cores} chunks to distribute among {n_cores} cores")

# Split all_uniprot_accn into chunks based on number of cores
chunks = np.array_split(all_uniprot_accn, n_cores)

def process_chunk(chunk, df_domainome_info):
    for uniprot_accn in chunk:
        map_plddt_for_uniprot(uniprot_accn, df_domainome_info)  # writes to disk, no need to collect return value

with tqdm(total=len([c for c in chunks if len(c) > 0]), desc="Processing chunks") as pbar:
    parallel = Parallel(n_jobs=n_cores, backend="multiprocessing")
    delayed_funcs = [delayed(process_chunk)(chunk, df_domainome_info) for chunk in chunks if len(chunk) > 0]
    for _ in parallel(delayed_funcs):
        pbar.update()

# All processed data has already been written to disk in each map_plddt_for_uniprot call, so nothing to keep in memory.

Splitting 18629 uniprot_accn into 72 chunks to distribute among 72 cores


Processing chunks: 100%|██████████| 72/72 [03:29<00:00,  2.90s/it]  


In [12]:
# concatenate all the csv files into one
df_domainome_info_uniprot_plddt = pd.concat([pd.read_csv(os.path.join(domain_info_dir, f"{uniprot_accn}.csv")) for uniprot_accn in all_uniprot_accn])

df_domainome_info_uniprot_plddt.iloc[0]

id                                                          Q8WVQ1-F1-nD1_A
uniprot_accn                                                         Q8WVQ1
Domain                                                                  nD1
Range                                                                86-400
ECOD_num                                                               2167
ECOD_key                                                            e1s1dA1
T-group                                                               5.1.2
DPAM_prob                                                             0.979
HH_prob                                                                 1.0
DALI_zscore                                                            57.6
Hit_cov                                                               0.994
Tgroup_cov                                                            0.969
Judge                                                           good_domain
Hcount      

In [ ]:
# Find rows whose n_failed is greater than 0 and sort descending by n_failed
df_domainome_info_uniprot_plddt[df_domainome_info_uniprot_plddt['n_failed'] > 0].sort_values(by='n_failed', ascending=False)

,id,uniprot_accn,Domain,Range,ECOD_num,ECOD_key,T-group,DPAM_prob,HH_prob,DALI_zscore,...,strand_frac,sasa,norm_sasa,protein_name,gene_name,domain_length,n_matched,n_failed,max_matched_dist,failed_matches


In [14]:
# Find whose whose max_matched_dist is greater than 0
df_domainome_info_uniprot_plddt[df_domainome_info_uniprot_plddt['max_matched_dist'] > 0]

,id,uniprot_accn,Domain,Range,ECOD_num,ECOD_key,T-group,DPAM_prob,HH_prob,DALI_zscore,...,strand_frac,sasa,norm_sasa,protein_name,gene_name,domain_length,n_matched,n_failed,max_matched_dist,failed_matches
